# Embeddings reduceren naar 2D met UMAP

Draait in Snowflake. Traint UMAP op de voorbeeldrapportages uit `NOTES` (`FRUTA.PUBLIC.NOTE_EMBEDDINGS`, zie `02_embeddings_local.ipynb`) en zet daarmee alle rapportages uit `RECORDS` op dezelfde 2D-kaart.

Resultaat:
- `FRUTA.PUBLIC.RECORD_UMAP_2D` (`NOTE_ID`, `X`, `Y`): wordt gebruikt door de app; `umap-learn` en het model zijn in de app niet nodig
- `FRUTA.PUBLIC.NOTE_UMAP_2D` (`ID`, `CATEGORY`, `X`, `Y`): de NOTES met hun categorie, als basis voor de achtergrondkaart
- het model op de stage `@FRUTA.PUBLIC.MODELS`, zodat later nieuwe rapportages op dezelfde kaart kunnen
- een plot per categorie, ook op die stage, en een interactieve versie waarin je over de punten kunt hoveren

Vereist de pakketten `umap-learn`, `matplotlib` en `plotly`: voeg ze toe via *Packages* (warehouse-runtime) of installeer ze met de cel hieronder (container-runtime).

In [ ]:
# Alleen nodig op container-runtime
!pip install umap-learn matplotlib plotly

In [ ]:
import json
import math
import os
import tempfile
import textwrap

import joblib
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
from snowflake.snowpark.context import get_active_session
from umap import UMAP

session = get_active_session()

NOTES_TABLE = "FRUTA.PUBLIC.NOTES"
NOTE_EMBEDDINGS_TABLE = "FRUTA.PUBLIC.NOTE_EMBEDDINGS"
NOTE_UMAP_TABLE = "FRUTA.PUBLIC.NOTE_UMAP_2D"
RECORD_EMBEDDINGS_TABLE = "FRUTA.PUBLIC.RECORD_EMBEDDINGS"
RECORD_UMAP_TABLE = "FRUTA.PUBLIC.RECORD_UMAP_2D"
MODEL_STAGE = "FRUTA.PUBLIC.MODELS"
MODEL_FILE = "notes_umap_2d.joblib"
PLOT_FILE = "notes_umap_2d.png"

UMAP_PARAMS = {
    "n_components": 2,  # Reduceert naar 2D voor visualisatie. 3D is ook mogelijk, maar moeilijker te interpreteren
    "n_neighbors": 12,  # Default: 15. Lagere waarden leiden tot meer lokale structuren, hogere waarden tot meer globale
    "min_dist": 0.05,  # Default: 0.1. Lagere waarde = dichtere clusters en meer gescheiden groepen
    "random_state": 6,  # Zorgt voor reproduceerbaarheid
}

In [ ]:
# Haalt embeddings op; VECTOR als ARRAY ophalen, dat komt in pandas binnen als JSON-tekst
def fetch_embeddings(sql):
    df = session.sql(sql).to_pandas()
    embs = np.array(
        [json.loads(e) if isinstance(e, str) else e for e in df["EMBEDDING"]],
        dtype=np.float32,
    )
    return df.drop(columns="EMBEDDING"), embs


notes_df, note_embs = fetch_embeddings(
    f"""
    SELECT n.ID, n.CATEGORY, n.NOTE, e.EMBEDDING::ARRAY AS EMBEDDING
    FROM {NOTE_EMBEDDINGS_TABLE} e
    JOIN {NOTES_TABLE} n ON n.ID = e.ID
    """
)
records_df, record_embs = fetch_embeddings(
    f"SELECT NOTE_ID, EMBEDDING::ARRAY AS EMBEDDING FROM {RECORD_EMBEDDINGS_TABLE}"
)
print(f"NOTES: {note_embs.shape}, RECORDS: {record_embs.shape}.")

In [ ]:
# Train UMAP op de NOTES en zet de RECORDS op dezelfde kaart
reducer = UMAP(**UMAP_PARAMS)
reduced_notes = reducer.fit_transform(note_embs)
reduced_records = reducer.transform(record_embs)

In [ ]:
# Schrijft een DataFrame naar FRUTA.PUBLIC; overschrijft de bestaande tabel
def write_table(df, table_name):
    session.write_pandas(
        df,
        table_name,
        database="FRUTA",
        schema="PUBLIC",
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=False,
    )
    print(f"Tabel '{table_name}' geschreven: {len(df)} rijen.")


df_notes_umap = notes_df[["ID", "CATEGORY"]].copy()
df_notes_umap["X"] = reduced_notes[:, 0]
df_notes_umap["Y"] = reduced_notes[:, 1]
write_table(df_notes_umap, "NOTE_UMAP_2D")

df_records_umap = records_df[["NOTE_ID"]].copy()
df_records_umap["X"] = reduced_records[:, 0]
df_records_umap["Y"] = reduced_records[:, 1]
write_table(df_records_umap, "RECORD_UMAP_2D")

In [ ]:
# Controle: aantal embeddings en aantal coördinaten
session.sql(
    f"""
    SELECT
        (SELECT COUNT(*) FROM {NOTE_EMBEDDINGS_TABLE}) AS NOTE_EMBEDDINGS,
        (SELECT COUNT(*) FROM {NOTE_UMAP_TABLE}) AS NOTE_COORDINATEN,
        (SELECT COUNT(*) FROM {RECORD_EMBEDDINGS_TABLE}) AS RECORD_EMBEDDINGS,
        (SELECT COUNT(*) FROM {RECORD_UMAP_TABLE}) AS RECORD_COORDINATEN
    """
).show()

## Model bewaren

Later weer inladen: `session.file.get(f"@{MODEL_STAGE}/{MODEL_FILE}", tempfile.gettempdir())` en dan `joblib.load(...)`.

In [ ]:
session.sql(f"CREATE STAGE IF NOT EXISTS {MODEL_STAGE}").collect()

model_path = os.path.join(tempfile.gettempdir(), MODEL_FILE)
joblib.dump(reducer, model_path)
session.file.put(model_path, f"@{MODEL_STAGE}", auto_compress=False, overwrite=True)
print(f"Model opgeslagen op @{MODEL_STAGE}/{MODEL_FILE}.")

## Plot per categorie

Basis voor de achtergrondkaart. Dezelfde assen als in de app: van het afgeronde minimum tot het afgeronde maximum van alle rapportages (RECORDS).

In [ ]:
x_range = [math.floor(reduced_records[:, 0].min()), math.ceil(reduced_records[:, 0].max())]
y_range = [math.floor(reduced_records[:, 1].min()), math.ceil(reduced_records[:, 1].max())]

fig, ax = plt.subplots(figsize=(10, 10))
ax.scatter(reduced_records[:, 0], reduced_records[:, 1], s=2, c="lightgray", label="rapportages (RECORDS)")
# Plotly-kleuren op alfabetische volgorde van de categorieën, net als in de app
categories = sorted(df_notes_umap["CATEGORY"].unique())
colors = px.colors.qualitative.Plotly
for i, category in enumerate(categories):
    group = df_notes_umap[df_notes_umap["CATEGORY"] == category]
    ax.scatter(group["X"], group["Y"], s=8, label=category, color=colors[i % len(colors)])
ax.set_xlim(x_range)
ax.set_ylim(y_range)
ax.grid(True, linewidth=0.5)
ax.legend(markerscale=3, loc="upper left", bbox_to_anchor=(1, 1))
ax.set_title(f"NOTES per categorie (x: {x_range}, y: {y_range})")

plot_path = os.path.join(tempfile.gettempdir(), PLOT_FILE)
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
session.file.put(plot_path, f"@{MODEL_STAGE}", auto_compress=False, overwrite=True)
print(f"Plot opgeslagen op @{MODEL_STAGE}/{PLOT_FILE}.")
plt.show()

In [ ]:
# Interactieve plot: NOTES gekleurd per categorie; hover over een punt om de rapportage te lezen
plot_df = df_notes_umap.merge(notes_df[["ID", "NOTE"]], on="ID")
plot_df["NOTE"] = plot_df["NOTE"].apply(lambda note: "<br>".join(textwrap.wrap(note, 60)))

fig = px.scatter(
    plot_df,
    x="X",
    y="Y",
    color="CATEGORY",
    hover_data={"NOTE": True, "CATEGORY": True, "X": False, "Y": False},
    range_x=x_range,
    range_y=y_range,
    height=800,
    title="NOTES per categorie",
    color_discrete_sequence=colors,
    category_orders={"CATEGORY": categories},
)
fig.update_traces(marker=dict(size=5, line=dict(width=0.5, color="DarkSlateGrey")))
fig.show()